# Notebook 2 : les embeddings et la recherche sémantique

Dans le premier notebook, la recherche comptait les mots identiques. Ici, on transforme les textes en vecteurs avec un modèle pré-entraîné, puis on compare leur proximité.

Le premier chargement du modèle d'embeddings télécharge ses poids. Les sections 1 à 8 fonctionnent sans LLM. La section 9 propose un appel facultatif à un LLM local avec Ollama : le modèle doit être téléchargé sur la machine, mais l'appel n'utilise pas d'API payante et le texte reste local.

## 1. L'idée

Un modèle d'embeddings transforme chaque texte en une liste de nombres (un vecteur). Les nombres ne sont pas des mots codés un par un : le modèle apprend une représentation qui permet souvent de rapprocher des formulations de sens voisin.

Nous comparerons les vecteurs avec la similarité cosinus. Une valeur plus élevée indique que les vecteurs pointent dans des directions plus proches. Ce score mesure une proximité selon le modèle, pas la vérité d'une affirmation.

In [2]:
pip install sentence-transformers

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------- ----------- 524.3/740.6 kB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 740.6/740.6 kB 3.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   -------------- ------------------------- 1.0/2.9 MB 6.2 MB/s eta 0:00:01
   ------------------------- -------------- 1.8/2.9 MB 5.1 MB/s eta 0:00:01
   ------------------------------------ --- 2.6/2.9 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 4.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.8/124.1 MB 5.1 MB/s eta 0:00:


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer

documents = [
    {
        "source": "note_llm",
        "text": (
            "Un LLM est un modèle entraîné à prédire la suite d'un texte. "
            "Il peut produire une réponse plausible qui contient pourtant une erreur."
        ),
    },
    {
        "source": "note_rag",
        "text": (
            "Le RAG recherche des passages dans des documents et les ajoute au contexte "
            "envoyé au LLM. Le RAG ne réentraîne pas le modèle."
        ),
    },
    {
        "source": "note_embeddings",
        "text": (
            "Un embedding représente un texte par une liste de nombres. "
            "On peut comparer ces représentations pour retrouver des textes proches par le sens."
        ),
    },
]

question = "Comment mesurer si deux formulations expriment une idée similaire ?"

for document in documents:
    print(f"[{document['source']}] {document['text']}")

print(f"\nQuestion : {question}")

c:\Users\lassi\projet\mlops\mlops_breast_cancer\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[note_llm] Un LLM est un modèle entraîné à prédire la suite d'un texte. Il peut produire une réponse plausible qui contient pourtant une erreur.
[note_rag] Le RAG recherche des passages dans des documents et les ajoute au contexte envoyé au LLM. Le RAG ne réentraîne pas le modèle.
[note_embeddings] Un embedding représente un texte par une liste de nombres. On peut comparer ces représentations pour retrouver des textes proches par le sens.

Question : Comment mesurer si deux formulations expriment une idée similaire ?


## 2. Comparaison avec la recherche par mots-clés

Cette baseline reprend l'idée du premier notebook. Elle ne reconnaît une relation que si des mots apparaissent à la fois dans la question et dans le document. On s'attend à ce qu'elle ne trouve pas la note pertinente pour notre formulation.

In [2]:
stop_words = {"le", "la", "les", "un", "une", "des", "de", "du", "et", "est", "au", "aux", "dans", "que", "qui", "quoi", "comment", "pourquoi", "il", "elle", "on", "ce", "se", "par", "en", "a", "à", "si", "deux", "une"}

def tokenize(text):
    words = re.findall(r"[a-zA-ZÀ-ÿ0-9]+", text.lower())
    return {word for word in words if word not in stop_words}

keyword_scores = []
question_words = tokenize(question)

for document in documents:
    score = len(question_words & tokenize(document["text"]))
    keyword_scores.append((score, document["source"]))

print("Scores par mots-clés :")
for score, source in sorted(keyword_scores, reverse=True):
    print(f"{source}: {score}")

Scores par mots-clés :
note_rag: 0
note_llm: 0
note_embeddings: 0


## 3. Charger un modèle d'embeddings

`sentence-transformers` fournit des modèles qui convertissent des phrases en vecteurs. Celui-ci est multilingue et convient à notre exemple français. Le téléchargement initial peut prendre un peu de temps ; le modèle sera ensuite mis en cache sur la machine.

In [3]:
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print("Modèle chargé.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 697.47it/s]


Modèle chargé.


## 4. Transformer les textes en vecteurs

Le modèle encode chaque texte complet en un seul vecteur : il ne crée pas ici un vecteur par mot. Ainsi, chaque ligne de `document_embeddings` représente un document entier, et `question_embedding` représente la question entière. Les nombres affichés sont les composantes de ces vecteurs, pas les mots du texte.

`normalize_embeddings=True` met chaque vecteur à une longueur de 1. Dans ce cas, leur produit scalaire est égal à leur similarité cosinus. `document_embeddings.shape[1]` indique le nombre de dimensions du vecteur, pas le nombre de mots.

In [4]:
document_texts = [document["text"] for document in documents]
document_embeddings = model.encode(
    document_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
question_embedding = model.encode(
    question,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print("Nombre de documents :", len(document_embeddings))
print("Taille d'un vecteur :", document_embeddings.shape[1])
print(document_embeddings)

Nombre de documents : 3
Taille d'un vecteur : 384
[[-0.13653189 -0.00383672 -0.05175284 ...  0.02564394  0.01527521
  -0.02329164]
 [-0.06857526  0.07563748  0.01812031 ... -0.02786318  0.05688829
  -0.01991608]
 [-0.06515794  0.00041913 -0.02516792 ...  0.00018107  0.06837747
   0.03855314]]


## 5. Calculer les similarités et classer les sources

Maintenant que la question et chaque document sont représentés par un vecteur de même taille, on peut les comparer. On compare le vecteur de la question successivement à celui de chaque document : cela donne un score de proximité pour chaque document. Comme les vecteurs ont été normalisés, le produit scalaire (`@`) calcule ici la similarité cosinus. Plus le score est élevé, plus le document est proche de la question selon le modèle.

On trie ensuite les documents du score le plus élevé au plus faible pour afficher en premier le passage le plus proche. Les scores exacts dépendent du modèle ; concentre-toi d'abord sur le classement.

In [5]:
similarities = document_embeddings @ question_embedding
ranked_results = sorted(
    zip(similarities, documents),
    key=lambda item: item[0],
    reverse=True,
)

for score, document in ranked_results:
    print(f"{score:.3f} | [{document['source']}] {document['text']}")

print("\nSource classée première :", ranked_results[0][1]["source"])

0.366 | [note_llm] Un LLM est un modèle entraîné à prédire la suite d'un texte. Il peut produire une réponse plausible qui contient pourtant une erreur.
0.320 | [note_embeddings] Un embedding représente un texte par une liste de nombres. On peut comparer ces représentations pour retrouver des textes proches par le sens.
0.101 | [note_rag] Le RAG recherche des passages dans des documents et les ajoute au contexte envoyé au LLM. Le RAG ne réentraîne pas le modèle.

Source classée première : note_llm


## 6. Utiliser le résultat pour préparer un prompt RAG

La recherche sémantique sélectionne ici le passage le plus proche. On l'ajoute à la question pour préparer l'entrée d'un LLM. Cette cellule ne génère pas la réponse : elle rend visible le contexte récupéré.

In [6]:
top_score, top_document = ranked_results[0]
prompt = (
    "Réponds à la question uniquement à partir de cette source. Si elle ne suffit pas, dis-le. "
    "Cite la source et n'invente pas d'information.\n\n"
    f"Source [{top_document['source']}] : {top_document['text']}\n\n"
    f"Question : {question}\nRéponse :"
)
print(prompt)

Réponds à la question uniquement à partir de cette source. Si elle ne suffit pas, dis-le. Cite la source et n'invente pas d'information.

Source [note_llm] : Un LLM est un modèle entraîné à prédire la suite d'un texte. Il peut produire une réponse plausible qui contient pourtant une erreur.

Question : Comment mesurer si deux formulations expriment une idée similaire ?
Réponse :


## 7. À retenir et à expérimenter

- Les mots-clés comparent les mots ; les embeddings comparent des représentations numériques apprises à partir du texte.
- Le classement sémantique est souvent plus tolérant aux formulations différentes, mais il peut aussi se tromper.
- Dans un RAG, la recherche récupère les sources ; le LLM rédige ensuite à partir du contexte fourni.

Essaie maintenant de modifier `question` dans la cellule 3, puis relance les cellules à partir de la recherche par mots-clés. Compare les deux classements. Essaie aussi une question dont la réponse ne figure pas dans les notes : un score élevé ne garantit pas que la source réponde vraiment à la question.

## 8. Récupérer plusieurs passages (top-k)

Jusqu'ici, nous avons gardé uniquement le passage le mieux classé. Pour une question qui nécessite plusieurs informations, on peut récupérer les `k` passages les plus proches. Ici, on choisit `k = 2` : le classement garde les deux premiers documents.

On rassemble ensuite leurs textes et leurs sources dans un même contexte. Dans un vrai RAG, ce contexte serait transmis au LLM. Nous allons seulement afficher le prompt préparé, sans appeler de modèle génératif. Augmenter `k` peut apporter plus d'informations, mais aussi ajouter des passages moins pertinents : il faut donc choisir cette valeur avec soin.

In [7]:
k = 2
selected_results = ranked_results[:k]

print(f"Les {len(selected_results)} passages retenus :")
for score, document in selected_results:
    print(f"\n{score:.3f} | [{document['source']}] {document['text']}")

retrieved_context = "\n\n".join(
    f"Source [{document['source']}] : {document['text']}"
    for score, document in selected_results
)
prompt_top_k = (
    "Réponds à la question uniquement à partir des sources ci-dessous. "
    "Si elles ne suffisent pas, dis-le et cite les sources utilisées.\n\n"
    f"{retrieved_context}\n\n"
    f"Question : {question}\nRéponse :"
)

print("\nPrompt préparé :\n")
print(prompt_top_k)

Les 2 passages retenus :

0.366 | [note_llm] Un LLM est un modèle entraîné à prédire la suite d'un texte. Il peut produire une réponse plausible qui contient pourtant une erreur.

0.320 | [note_embeddings] Un embedding représente un texte par une liste de nombres. On peut comparer ces représentations pour retrouver des textes proches par le sens.

Prompt préparé :

Réponds à la question uniquement à partir des sources ci-dessous. Si elles ne suffisent pas, dis-le et cite les sources utilisées.

Source [note_llm] : Un LLM est un modèle entraîné à prédire la suite d'un texte. Il peut produire une réponse plausible qui contient pourtant une erreur.

Source [note_embeddings] : Un embedding représente un texte par une liste de nombres. On peut comparer ces représentations pour retrouver des textes proches par le sens.

Question : Comment mesurer si deux formulations expriment une idée similaire ?
Réponse :


## 9. Faire rédiger une réponse par un LLM local (gratuit)

Les embeddings servent à retrouver les passages pertinents ; ils ne rédigent pas la réponse. Chaque phrase ou document encodé est représenté par un vecteur, pas par une valeur numérique par mot. Le LLM reçoit le texte du prompt préparé à l'étape précédente (`prompt_top_k`), c'est-à-dire la question et les passages retenus, mais pas les vecteurs.

Nous utilisons `qwen2.5:3b` avec Ollama. Il faut installer l'application Ollama pour Windows : `%pip install ollama` installe uniquement la bibliothèque Python cliente, pas le serveur. Après l'installation, démarre Ollama puis lance `ollama pull qwen2.5:3b` dans PowerShell pour télécharger le modèle. Le modèle est gratuit, mais utilise de l'espace disque et les ressources de ton ordinateur. Le texte est traité localement, sans clé ni facturation par requête.

In [9]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from urllib.error import URLError
from urllib.request import urlopen

from ollama import chat

try:
    with urlopen("http://localhost:11434/api/tags", timeout=3):
        pass
except (URLError, TimeoutError) as error:
    raise RuntimeError(
        "Serveur Ollama inaccessible. Installe l'application Ollama pour Windows, "
        "démarre-la, puis relance cette cellule."
    ) from error

response = chat(
    model="qwen2.5:3b",
    messages=[{"role": "user", "content": prompt_top_k}],
)

print(response.message.content)

Pour mesurer si deux formulations expriment une idée similaire, on peut utiliser des techniques comme les embeddings. Les embeddings peuvent être utilisés pour représenter les textes comme des listes de nombres, ce qui permet de comparer ces représentations pour déterminer si deux formulations sont proches en termes de sens. Par exemple, si deux formulations ont des embeddings très similaires, cela pourrait indiquer que les deux expriment une idée similaire. Cependant, il est important de noter que même si deux formulations ont des embeddings proches, cela ne garantit pas nécessairement qu'elles expriment exactement la même idée, vu que les embeddings peuvent être influencées par de nombreux facteurs, tels que le contexte, la terminologie spécifique utilisée, et d'autres aspects.

Dans le cadre de cette réponse, j'ai utilisé la source [note_embeddings] pour expliquer comment les embeddings peuvent être utilisés pour comparer des formulations.
